In [69]:
from fpl_project.scripts import config
from fpl_project.scripts.data_utils.features_config import FeaturesConfig
from fpl_project.scripts.model_utils.data import FPLDataPipe, train_test_split
from fpl_project.scripts.model_utils.trainer import set_seed
from fpl_project.scripts.model_utils.selector import ModelSelector, initialize_device

from typing import NamedTuple
from dataclasses import dataclass
from typing import Any, Dict
from fpl_project.scripts.model_utils.models import FPLLoss
import pandas as pd
import numpy as np

In [70]:
set_seed(77)
if not config.FEATURES_ENGINEERING.exists():
    from fpl_project.scripts.data_utils.features_processing import build_and_save_features_engineering_data
    build_and_save_features_engineering_data()
data = pd.read_parquet(config.FEATURES_ENGINEERING)
data.total_points = data.total_points.clip(lower=0)

In [71]:
train_season, test_season = 2526, 2627
print(f"Train season: {train_season} (rows: {(data.season == train_season).sum()})")
print(f"Test season:  {test_season} (rows: {(data.season == test_season).sum()})")

Train season: 2526 (rows: 29338)
Test season:  2627 (rows: 747)


In [72]:
played_mask = data["is_played"].astype(bool) if "is_played" in data.columns else data.total_points.notna()
train = data[(data.season == train_season) & played_mask].copy()
test = data[(data.season == test_season) & played_mask].copy()
predict_df = data[~played_mask].copy()

cols_map = FeaturesConfig()

num_cols = [c for c in data.columns if c not in cols_map.id_cols + cols_map.static_cols + cols_map.target]
cat_cols = cols_map.static_cols

In [73]:
preds = pd.read_parquet(config.ML_PREDS_PATH)

preds_test_dict = {}
preds_next_dict = {}
for algo_name, algo_path in [("glm", config.PREDS_GLM), ("xgboost", config.PREDS_XGBOOST), ("mlp", config.PREDS_MLP), ("conv1d", config.PREDS_CONV1D)]:
    if algo_path.exists():
        df_algo = pd.read_parquet(algo_path)
        preds_test_dict[algo_name] = df_algo.loc[df_algo["is_played"], algo_name].clip(lower=0, upper=15).values
        preds_next_dict[algo_name] = df_algo.loc[~df_algo["is_played"], algo_name].clip(lower=0, upper=15).values

preds_test = pd.DataFrame(preds_test_dict)
preds_next_gw_models = pd.DataFrame(preds_next_dict)

print(f"Preds train shape: {preds.shape}")
print(f"Preds test shape:  {preds_test.shape}")


Preds train shape: (29338, 2)
Preds test shape:  (747, 2)


In [74]:
fpl_pipe = FPLDataPipe(num_cols=num_cols, 
                       cat_cols=cat_cols, 
                       catboost_flag=False)

fpl_pipe.preprocessor.fit(train)
X_train_base = fpl_pipe.preprocessor.transform(train).reset_index(drop=True)
X_test_base = fpl_pipe.preprocessor.transform(test).reset_index(drop=True)

In [75]:
X_train_full = pd.concat([X_train_base, preds.reset_index(drop=True)], axis=1)
X_test_full = pd.concat([X_test_base, preds_test.reset_index(drop=True)], axis=1)

selected_cols = [c for c in X_train_full.columns if 'total_points' in c or c in ['glm', 'xgboost']]
X_train = X_train_full[selected_cols].copy()
X_test = X_test_full[selected_cols].copy()

y_train = train["total_points"].reset_index(drop=True)
y_test = test["total_points"].reset_index(drop=True)

In [76]:
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}  | y_test shape:  {y_test.shape}")
print(f"Nulls in X_train: {X_train.isna().sum().sum()} | Nulls in X_test: {X_test.isna().sum().sum()}")
print(f"Wybrane kolumny ({len(selected_cols)}): {list(X_train.columns)}")

X_train shape: (29338, 8) | y_train shape: (29338,)
X_test shape:  (747, 8)  | y_test shape:  (747,)
Nulls in X_train: 0 | Nulls in X_test: 0
Wybrane kolumny (8): ['num__total_points_ema_3', 'num__total_points_lagged_1', 'num__total_points_lagged_3', 'num__total_points_lagged_5', 'num__total_points_std_3', 'num__total_points_std_5', 'glm', 'xgboost']


In [77]:
from glum import GeneralizedLinearRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [78]:
rf_reg = RandomForestRegressor(
    n_estimators=150,
    max_depth=5,
    min_samples_leaf=20,
    min_samples_split=50,
    random_state=77,
    n_jobs=-1
)
rf_reg.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",150
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",50
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",20
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",77
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"ma

In [79]:
glm_meta = GeneralizedLinearRegressor(
    family="normal",
    alpha=0.01,
    l1_ratio=0.0,
    random_state=77
)
glm_meta.fit(X_train, y_train)

,alpha,0.01
,random_state,77
,l1_ratio,0.0
,P1,'identity'
,P2,'identity'
,fit_intercept,True
,family,'normal'
,link,'auto'
,solver,'auto'
,max_iter,100
,max_inner_iter,100000


In [80]:
models = {
    "RandomForest (Regularized)": rf_reg,
    "GLM (Ridge Normal)": glm_meta
}

metrics_list = []
for name, model in models.items():
    p_train = model.predict(X_train)
    p_test = model.predict(X_test)
    if "GLM" in name:
        p_test = np.clip(p_test, 0, 15)
    metrics_list.append({
        "Model": name,
        "Train MSE": mean_squared_error(y_train, p_train),
        "Train MAE": mean_absolute_error(y_train, p_train),
        "Train R2": r2_score(y_train, p_train),
        "Test MSE": mean_squared_error(y_test, p_test),
        "Test MAE": mean_absolute_error(y_test, p_test),
        "Test R2": r2_score(y_test, p_test)
    })

results_df = pd.DataFrame(metrics_list)
results_df

,Model,Train MSE,Train MAE,Train R2,Test MSE,Test MAE,Test R2
0,RandomForest (Regularized),3.59549,0.976791,0.348484,6.560475,1.710712,0.101240
1,GLM (Ridge Normal),3.68277,1.001575,0.332669,7.034248,1.727941,0.036335


In [81]:
rf_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": rf_reg.feature_importances_
}).sort_values(by="importance", ascending=False)
rf_importance

,feature,importance
7,xgboost,0.966418
6,glm,0.015681
0,num__total_points_ema_3,0.007135
5,num__total_points_std_5,0.002775
3,num__total_points_lagged_5,0.002398
2,num__total_points_lagged_3,0.002326
4,num__total_points_std_3,0.002109
1,num__total_points_lagged_1,0.001158


In [82]:
glm_importance = pd.DataFrame({
    "feature": X_train.columns,
    "coef": glm_meta.coef_
}).sort_values(by="coef", key=abs, ascending=False)
glm_importance

,feature,coef
7,xgboost,0.934873
6,glm,0.145110
4,num__total_points_std_3,-0.070468
5,num__total_points_std_5,-0.051462
2,num__total_points_lagged_3,0.045152
3,num__total_points_lagged_5,0.044817
0,num__total_points_ema_3,0.014648
1,num__total_points_lagged_1,-0.011243


In [ ]:
meta_cols = ["name", "code", "team", "opponent_team", "position", "was_home", "season", "gw", "kickoff_time"]
test_meta = test[meta_cols].copy().reset_index(drop=True)
test_meta["is_played"] = True

if not predict_df.empty and not preds_next_gw_models.empty:
    X_pred_base = fpl_pipe.preprocessor.transform(predict_df).reset_index(drop=True)
    X_pred_full = pd.concat([X_pred_base, preds_next_gw_models.reset_index(drop=True)], axis=1)
    X_pred_meta = X_pred_full[selected_cols].copy()
    pred_meta = predict_df[meta_cols].copy().reset_index(drop=True)
    pred_meta["is_played"] = False
    season_2627_meta = pd.concat([test_meta, pred_meta], ignore_index=True)
else:
    X_pred_meta = None
    season_2627_meta = test_meta

for meta_name, meta_model, meta_path in [("rf_meta", rf_reg, config.PREDS_RF_META), ("glm_meta", glm_meta, config.PREDS_GLM_META)]:
    out_df = season_2627_meta.copy()
    p_test = np.clip(meta_model.predict(X_test), 0, 15)
    if X_pred_meta is not None:
        p_next = np.clip(meta_model.predict(X_pred_meta), 0, 15)
        out_df[meta_name] = np.concatenate([p_test, p_next])
    else:
        out_df[meta_name] = p_test
    out_df.to_parquet(meta_path, index=False)
    print(f"Saved {meta_name}: {out_df.shape} -> {meta_path}")
